In [1]:
pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 70.7 MB/s eta 0:00:00


In [2]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 51.3 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
import os
import json
import pymupdf  # PyMuPDF
import pdfplumber
import pandas as pd
from PIL import Image
from pathlib import Path
from langchain_core.documents import Document

In [21]:
import os

directory_name = "output_ragparsing"

# Check if the directory already exists to avoid an error
if not os.path.exists(directory_name):
    os.makedirs(directory_name)
    print(f"Directory '{directory_name}' created successfully.")
else:
    print(f"Directory '{directory_name}' already exists.")

Directory 'output_ragparsing' created successfully.


In [22]:
pdf_path = "/content/Complex_pdf.pdf"
output_dir = Path("/content/output_ragparsing")
img_dir = output_dir/"extracted_images"
page_img_dir = output_dir/"page_images"
img_dir.mkdir(parents=True, exist_ok=True)
page_img_dir.mkdir(parents=True, exist_ok=True)

print("PDF exists:", os.path.exists(pdf_path))

PDF exists: True


In [15]:
!pip install pytesseract

In [23]:
import os
import pytesseract
from PIL import Image

# In Colab, Tesseract is typically in the system PATH and doesn't need explicit setting.
# The previous path 'C:\Program Files\Tesseract-OCR\tesseract.exe' is for Windows.
# If you encounter issues, you might need to specify a path like '/usr/bin/tesseract'
# but it's often not necessary.

print("Tesseract version:", pytesseract.get_tesseract_version())

Tesseract version: 4.1.1


In [24]:
def run_ocr_on_image(image_path):
    """
    Runs OCR on image using pytesseract.
    If tesseract is not installed in system, it will return empty text.
    """
    try:
        import pytesseract
        img = Image.open(image_path)
        text = pytesseract.image_to_string(img)
        return text.strip()
    except Exception as e:
        return f"[OCR_SKIPPED_OR_FAILED: {str(e)}]"

In [25]:
def extract_text_and_images(pdf_path):
    doc = pymupdf.open(pdf_path)

    page_records = []
    image_records = []

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_number = page_index + 1

        # Extract normal selectable text
        text = page.get_text("text")

        # Extract page metadata-like info
        page_info = {
            "page_number": page_number,
            "text": text.strip(),
            "image_count": len(page.get_images(full=True)),
            "width": page.rect.width,
            "height": page.rect.height,
        }

        page_records.append(page_info)

        # Render full page as image for OCR
        pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))
        page_image_path = page_img_dir / f"page_{page_number:03d}.png"
        pix.save(str(page_image_path))

        # OCR full page image
        ocr_text = run_ocr_on_image(page_image_path)
        page_info["ocr_text"] = ocr_text
        page_info["page_image_path"] = str(page_image_path)

        # Extract embedded images
        images = page.get_images(full=True)

        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)

            image_bytes = base_image["image"]
            image_ext = base_image["ext"]

            image_path = img_dir / f"page_{page_number:03d}_image_{img_index + 1}.{image_ext}"

            with open(image_path, "wb") as f:
                f.write(image_bytes)

            image_ocr_text = run_ocr_on_image(image_path)

            image_records.append({
                "page_number": page_number,
                "image_index": img_index + 1,
                "image_path": str(image_path),
                "image_ext": image_ext,
                "image_ocr_text": image_ocr_text
            })

    doc.close()

    return page_records, image_records

In [26]:
page_records, image_records = extract_text_and_images(pdf_path)

print("Total pages parsed:", len(page_records))
print("Total images extracted:", len(image_records))

Total pages parsed: 24
Total images extracted: 13


### Extracting Tables with `pdfplumber`

Now, let's use `pdfplumber` to extract tables from the PDF. `pdfplumber` is excellent for structured data extraction. I will demonstrate extracting tables from page 2.

### Extracting All Tables

Now, let's extract tables from *all* pages of the PDF and store them for further analysis.

In [27]:
def extract_tables(pdf_path):
    table_records = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_index, page in enumerate(pdf.pages):
            page_number = page_index + 1

            try:
                tables = page.extract_tables()
            except Exception as e:
                tables = []
                print(f"Table extraction failed on page {page_number}: {e}")

            for table_index, table in enumerate(tables):
                if not table:
                    continue

                # Clean table rows
                cleaned_table = []
                for row in table:
                    cleaned_row = [
                        cell.strip() if isinstance(cell, str) else cell
                        for cell in row
                    ]
                    cleaned_table.append(cleaned_row)

                # Convert to DataFrame safely
                try:
                    df = pd.DataFrame(cleaned_table[1:], columns=cleaned_table[0])
                except Exception:
                    df = pd.DataFrame(cleaned_table)

                table_records.append({
                    "page_number": page_number,
                    "table_index": table_index + 1,
                    "raw_table": cleaned_table,
                    "markdown": df.to_markdown(index=False),
                    "csv": df.to_csv(index=False)
                })

    return table_records


table_records = extract_tables(pdf_path)

print("Total tables extracted:", len(table_records))

for table in table_records[:3]:
    print("\nPage:", table["page_number"], "Table:", table["table_index"])
    print(table["markdown"][:1000])

Total tables extracted: 16

Page: 1 Table: 1
| Section               | Parsing challenge                       | Why it matters for RAG                  |
|:----------------------|:----------------------------------------|:----------------------------------------|
| Contracts             | Dense legal text, clause numbers, cross | Need section-aware chunks and citations |
|                       | references                              |                                         |
| Tables                | Merged headers, numeric columns,        | Need row/column preservation            |
|                       | footnotes                               |                                         |
| Images                | Architecture diagram, heatmap, scanned  | Need OCR or multimodal extraction       |
|                       | form                                    |                                         |
| Multi-tenant metadata | client_id, document_id,                 | Need ac

In [28]:
langchain_docs = []

# 5.1 Page text documents
for page in page_records:
    page_number = page["page_number"]

    combined_text = f"""
PAGE {page_number}

SELECTABLE TEXT:
{page["text"]}

OCR TEXT:
{page["ocr_text"]}
""".strip()

    doc = Document(
        page_content=combined_text,
        metadata={
            "source": pdf_path,
            "page_number": page_number,
            "content_type": "page_text_plus_ocr",
            "image_count": page["image_count"],
            "page_image_path": page["page_image_path"],
        }
    )

    langchain_docs.append(doc)


# 5.2 Table documents
for table in table_records:
    page_number = table["page_number"]

    table_text = f"""
TABLE FOUND ON PAGE {page_number}
TABLE INDEX: {table["table_index"]}

TABLE MARKDOWN:
{table["markdown"]}
""".strip()

    doc = Document(
        page_content=table_text,
        metadata={
            "source": pdf_path,
            "page_number": page_number,
            "content_type": "table",
            "table_index": table["table_index"],
        }
    )

    langchain_docs.append(doc)

In [29]:
for image in image_records:
    page_number = image["page_number"]

    image_text = f"""
IMAGE FOUND ON PAGE {page_number}
IMAGE INDEX: {image["image_index"]}
IMAGE PATH: {image["image_path"]}

IMAGE OCR TEXT:
{image["image_ocr_text"]}
""".strip()

    doc = Document(
        page_content=image_text,
        metadata={
            "source": pdf_path,
            "page_number": page_number,
            "content_type": "image",
            "image_index": image["image_index"],
            "image_path": image["image_path"],
            "image_ext": image["image_ext"],
        }
    )

    langchain_docs.append(doc)


print("Total LangChain Documents created:", len(langchain_docs))

Total LangChain Documents created: 53


In [30]:
# Save page records
with open(output_dir / "page_records.json", "w", encoding="utf-8") as f:
    json.dump(page_records, f, indent=2, ensure_ascii=False)

# Save image records
with open(output_dir / "image_records.json", "w", encoding="utf-8") as f:
    json.dump(image_records, f, indent=2, ensure_ascii=False)

# Save table records
with open(output_dir / "table_records.json", "w", encoding="utf-8") as f:
    json.dump(table_records, f, indent=2, ensure_ascii=False)

# Save all LangChain document content as markdown
with open(output_dir / "rag_ready_documents.md", "w", encoding="utf-8") as f:
    for i, doc in enumerate(langchain_docs):
        f.write(f"\n\n# Document {i + 1}\n")
        f.write(f"\nMetadata:\n```json\n{json.dumps(doc.metadata, indent=2)}\n```\n")
        f.write("\nContent:\n")
        f.write(doc.page_content)
        f.write("\n\n---\n")

# Save table markdown separately
with open(output_dir / "extracted_tables.md", "w", encoding="utf-8") as f:
    for table in table_records:
        f.write(f"\n\n## Page {table['page_number']} - Table {table['table_index']}\n\n")
        f.write(table["markdown"])
        f.write("\n\n---\n")

print("\nSaved outputs in:", output_dir)


Saved outputs in: /content/output_ragparsing


In [31]:
print("\n================ PAGE TEXT PREVIEW ================\n")
print(langchain_docs[0].page_content[:1500])

print("\n================ TABLE PREVIEW ================\n")
table_docs = [doc for doc in langchain_docs if doc.metadata["content_type"] == "table"]

if table_docs:
    print(table_docs[0].page_content[:1500])
else:
    print("No table docs found.")

print("\n================ IMAGE OCR PREVIEW ================\n")
image_docs = [doc for doc in langchain_docs if doc.metadata["content_type"] == "image"]

if image_docs:
    print(image_docs[0].page_content[:1500])
else:
    print("No image docs found.")


================ PAGE TEXT PREVIEW ================

PAGE 1

SELECTABLE TEXT:
Complex RAG Parsing Sample - synthetic document
Page 1
Complex Document for RAG Parsing Tests
Synthetic 15-page PDF with paragraphs, simple and complex tables, diagrams, scanned-form style image, metadata
examples, and production RAG edge cases.
Story Line
Three client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records,
and operational reports into a single RAG platform. Each team has different document types, access rules, and parsing
challenges. The RAG system must answer questions with citations while ensuring that one client never sees another clients
data.
This PDF is intentionally designed to test document loaders, PDF parsers, OCR workflows, table extraction, chunking
strategies, metadata preservation, and source citation quality.
Key statement: RAG does not train the model. RAG gives the model the right context before answering.
Section
P

In [2]:
!pip install -q docling langchain-core langchain-text-splitters pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 814.7/814.7 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.0/300.0 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 125.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.

In [3]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
import os

directory_name = "output_ragparsing_docling"

# Check if the directory already exists to avoid an error
if not os.path.exists(directory_name):
    os.makedirs(directory_name)
    print(f"Directory '{directory_name}' created successfully.")
else:
    print(f"Directory '{directory_name}' already exists.")

Directory 'output_ragparsing_docling' created successfully.


In [5]:
PDF_PATH = Path("/content/Complex_pdf.pdf")

OUTPUT_DIR = Path("/content/output_ragparsing_docling")


print("PDF path:", PDF_PATH)

PDF path: /content/Complex_pdf.pdf


In [6]:
!pip install --upgrade Pillow --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 83.0 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [7]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

# Convert PDF into Docling structured document
conversion_result = converter.convert(PDF_PATH)

# Main Docling document object
docling_doc = conversion_result.document

print("Docling conversion completed.")

[INFO] 2026-09-07 13:54:22,392 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-07 13:54:22,399 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-07 13:54:22,402 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.2/torch/PP-OCRv6/det/PP-OCRv6_det_small.pth
[INFO] 2026-09-07 13:54:24,813 [RapidOCR] download_file.py:82: Download size: 9.77MB
[INFO] 2026-09-07 13:54:25,183 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-07 13:54:25,186 [RapidOCR] main.py:50: Using /usr/local/lib/python3.13/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-09-07 13:54:25,645 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-07 13:54:25,646 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-09-07 13:54:25,648 [RapidOCR] download_file.py:68: Initiating downlo

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1024.)
  return F.conv2d(


Docling conversion completed.


In [8]:
print(type(docling_doc))

<class 'docling_core.types.doc.document.DoclingDocument'>


In [14]:
# Export the document to Markdown (Recommended)
markdown_content = docling_doc.export_to_markdown()
print(markdown_content[:800])

## Complex Document for RAG Parsing Tests

Synthetic  15-page  PDF  with  paragraphs,  simple  and  complex  tables,  diagrams,  scanned-form  style  image,  metadata examples, and production RAG edge cases.

## Story Line

Three client teams - Arka Finance, BlueLeaf Retail, and CityRide Mobility - are migrating contracts, policies, support records, and  operational  reports  into  a  single  RAG  platform.  Each  team  has  different  document  types,  access  rules,  and  parsing challenges. The RAG system must answer questions with citations while ensuring that one client never sees another clients data.

This  PDF  is  intentionally  designed  to  test  document  loaders,  PDF  parsers,  OCR  workflows,  table  extraction,  chunking strategies, metadata preservation, and source citatio
